In [3]:
# Load environment variables from .env
import os
from dotenv import load_dotenv

load_dotenv("keys.env")

print("Gemini key present?   ", bool(os.getenv("GEMINI_API_KEY")))
print("OpenAI key present?   ", bool(os.getenv("OPENAI_API_KEY")))
print("Anthropic key present?", bool(os.getenv("ANTHROPIC_API_KEY")))


Gemini key present?    True
OpenAI key present?    True
Anthropic key present? True


### Step 1: Gemini streaming (low-latency)

In [12]:
import time
from google import genai
from google.genai import types

In [ ]:
# create Gemini client (uses GEMINI_API_KEY from env)
g_client = genai.Client()

: 

In [ ]:

def stream_gemini(prompt: str,
                  model: str = "gemini-2.5-flash",
                  temperature: float = 0.7,
                  max_output_tokens: int = 256,
                  thinking_budget: int = 0):
    """Stream response text from Gemini with low latency."""
    t0 = time.perf_counter()
    first = None
    parts = []

    stream = g_client.models.generate_content_stream(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget)
        ),
    )

    for chunk in stream:
        if chunk.text:
            if first is None:
                first = time.perf_counter()
            print(chunk.text, end="", flush=True)
            parts.append(chunk.text)

    print()
    return {
        "provider": "gemini",
        "model": model,
        "ttfb_s": None if first is None else first - t0,
        "total_s": time.perf_counter() - t0,
        "text": "".join(parts),
    }


In [11]:
result = stream_gemini("Hi")
print("\nTiming:", result)


Hello! How can I help you today?

Timing: {'provider': 'gemini', 'model': 'gemini-2.5-flash', 'ttfb_s': 0.6270120999979554, 'total_s': 0.7010693000047468, 'text': 'Hello! How can I help you today?'}
